# Week 6 - Transformer encoder-decoder sederhana
Marvel Kevin Nathanael | 00000108042

Praktik positional encoding, self-attention kausal, cross-attention, dan prediksi token berikutnya dari TLM 6. Pasangan Portugis-Inggris sintetis ini hanya untuk uji mekanisme.

In [ ]:
import itertools
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)
subjects = [("eu", "i"), ("voce", "you"), ("ele", "he"), ("ela", "she")]
verbs = [("gosto", "like"), ("vejo", "see"), ("quero", "want")]
objects = [("agua", "water"), ("cafe", "coffee"), ("leite", "milk")]
pairs = [(f"{s[0]} {v[0]} {o[0]}", f"{s[1]} {v[1]} {o[1]}")
         for s, v, o in itertools.product(subjects, verbs, objects)]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
train_pairs, val_pairs = train_test_split(train_pairs, test_size=0.2, random_state=42)
src_vocab = ["<pad>", "<unk>"] + sorted({t for s, _ in train_pairs for t in s.split()})
tgt_vocab = ["<pad>", "<unk>", "<start>", "<end>"] + sorted(
    {t for _, s in train_pairs for t in s.split()})
src_to_id = {t: i for i, t in enumerate(src_vocab)}
tgt_to_id = {t: i for i, t in enumerate(tgt_vocab)}
id_to_tgt = dict(enumerate(tgt_vocab))
def arrays(data):
    src = np.array([[src_to_id.get(t, 1) for t in a.split()]
                    for a, _ in data], dtype="int32")
    target = np.array([[tgt_to_id.get(t, 1) for t in b.split()] + [tgt_to_id["<end>"]]
                       for _, b in data], dtype="int32")
    dec_in = np.concatenate([
        np.full((len(data), 1), tgt_to_id["<start>"], dtype="int32"),
        target[:, :-1]], axis=1)
    return (src, dec_in), target
(x_train, y_train), (x_val, y_val), (x_test, y_test) = map(
    arrays, [train_pairs, val_pairs, test_pairs])
print("Train/val/test:", len(train_pairs), len(val_pairs), len(test_pairs))

In [ ]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model, max_len=16):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        pos = np.arange(max_len)[:, None]
        depth = np.arange(d_model)[None, :]
        angles = pos / np.power(10000.0, (2 * (depth // 2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        self.position = tf.constant(angles[None, ...], dtype=tf.float32)
        self.scale = np.sqrt(d_model)

    def call(self, tokens):
        x = self.embedding(tokens) * self.scale
        return x + self.position[:, :tf.shape(tokens)[1], :]

D_MODEL = 64
src = tf.keras.Input(shape=(None,), dtype=tf.int32, name="portuguese")
dec = tf.keras.Input(shape=(None,), dtype=tf.int32, name="english_prefix")
x = PositionalEmbedding(len(src_vocab), D_MODEL)(src)
attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
x = tf.keras.layers.LayerNormalization()(x + attn)
ffn = tf.keras.layers.Dense(D_MODEL, activation="relu")(x)
x = tf.keras.layers.LayerNormalization()(x + ffn)

y = PositionalEmbedding(len(tgt_vocab), D_MODEL)(dec)
self_attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(
    y, y, use_causal_mask=True)
y = tf.keras.layers.LayerNormalization()(y + self_attn)
cross = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(y, x)
y = tf.keras.layers.LayerNormalization()(y + cross)
ffn = tf.keras.layers.Dense(D_MODEL, activation="relu")(y)
y = tf.keras.layers.LayerNormalization()(y + ffn)
logits = tf.keras.layers.Dense(len(tgt_vocab))(y)
model = tf.keras.Model([src, dec], logits)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["sparse_categorical_accuracy"])
model.summary()

In [ ]:
history = model.fit(
    x_train, y_train, validation_data=(x_val, y_val),
    epochs=150, batch_size=8, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True)])
print("Epoch terpakai:", len(history.history["loss"]))

def translate(sentence):
    source = np.array([[src_to_id.get(t, 1) for t in sentence.split()]], dtype="int32")
    prefix = [tgt_to_id["<start>"]]
    for _ in range(4):
        pred = model.predict([source, np.array([prefix], dtype="int32")], verbose=0)
        token = int(np.argmax(pred[0, -1]))
        if token == tgt_to_id["<end>"]:
            break
        prefix.append(token)
    return " ".join(id_to_tgt[i] for i in prefix[1:])

for src_text, expected in test_pairs:
    print(f"{src_text:20s} -> {translate(src_text):20s} | target: {expected}")
exact = np.mean([translate(a) == b for a, b in test_pairs])
print(f"Exact-match test: {exact:.3f}")

**Catatan:** Causal mask mencegah decoder melihat token berikutnya saat training. Test hanya berisi kombinasi kosakata yang sama; exact match tidak mewakili terjemahan dunia nyata.